# Aktywacja Strategii Odbicie Ogórkowe
Ten notatnik sĹ‚uĹĽy do testowania, wizualizacji i optymalizacji strategii powrotu do Ĺ›redniej po mocnych spadkach.

## Importy, dane i sygnały

In [1]:
import os
import sys

# Dodajemy folder glowny do path aby moduly dzialaly
sys.path.append(os.path.abspath('c:/Users/PC/Documents/Antigravity/lyse-lby'))
os.chdir('c:/Users/PC/Documents/Antigravity/lyse-lby')

import optuna
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

# Importujemy ladowanie danych i nasze nowe moduly
from core.ladowanie_danych import create_stock_dfs
from odbicie.mackowe_sygnaly import mackowe_sygnaly
from odbicie.strategie.odbicie import generate_odbicie_entries
from odbicie.odbicie_atr import generate_odbicie_atr_entries
from odbicie.odbicie_bb import generate_odbicie_bb_entries
from odbicie.tbm.tbm import moving_triple_barrier_labels
from odbicie.optymalizacja import optimize_atr_tbm, optimize_bb_tbm

# Katalog na pliki cache - zawsze wewnatrz odbicie/dane/, niezalezny od cwd
# __vsc_ipynb_file__ dostepny w VS Code; fallback na abspath('')
_SCRIPT_DIR = os.path.dirname(os.path.abspath(globals()["__vsc_ipynb_file__"])) if "__vsc_ipynb_file__" in globals() else os.path.abspath("")
# Jesli notebook lezy wewnatrz odbicie/, katalog jest katalogiem rodzica
if os.path.basename(_SCRIPT_DIR) != 'odbicie':
    _SCRIPT_DIR = os.path.join(_SCRIPT_DIR, 'odbicie')
DANE_DIR = os.path.join(_SCRIPT_DIR, 'cache')
os.makedirs(DANE_DIR, exist_ok=True)
print(f"Cache dir: {DANE_DIR}")

# Ustawienia ladowania danych
import json
with open(os.path.join(_SCRIPT_DIR, 'settings.json'), 'r') as f:
    all_settings = json.load(f)

# Ustawienia ladowania danych
settings = all_settings['data_settings']

Cache dir: c:\Users\PC\Documents\Antigravity\lyse-lby\odbicie\cache


In [2]:
# 1. Ladowanie Danych
import pickle

market = settings.get('market', 'all')
data_cache_file = os.path.join(DANE_DIR, f'dfs_cache_{market}.pkl')

if os.path.exists(data_cache_file):
    print("Znaleziono zapisane dane. Wczytywanie z pliku...")
    with open(data_cache_file, "rb") as f:
        dfs_1d, dfs_1w = pickle.load(f)
    print(f"Wczytano {len(dfs_1d)} symboli 1D i {len(dfs_1w)} symboli 1W z pliku {data_cache_file}.")
else:
    print("Ladowanie danych dziennych i tygodniowych...")
    dfs_1d, dfs_1w = create_stock_dfs(settings)
    print(f"Zaladowano {len(dfs_1d)} symboli 1D i {len(dfs_1w)} symboli 1W.")
    print("Zapisywanie danych do pliku...")
    with open(data_cache_file, "wb") as f:
        pickle.dump((dfs_1d, dfs_1w), f)
    print("Dane zapisane pomyslnie.")

Znaleziono zapisane dane. Wczytywanie z pliku...
Wczytano 478 symboli 1D i 478 symboli 1W z pliku c:\Users\PC\Documents\Antigravity\lyse-lby\odbicie\cache\dfs_cache_sp500.pkl.


In [3]:
# 2. Generowanie Sygnalow Bazowych (mackowe_sygnaly)
market = settings.get('market', 'all')
signals_cache_file = os.path.join(DANE_DIR, f'signals_cache_{market}.pkl')

if os.path.exists(signals_cache_file):
    print("Znaleziono zapisane sygnaly. Wczytywanie z pliku...")
    with open(signals_cache_file, "rb") as f:
        signals_df = pickle.load(f)
    print(f"Wczytano {len(signals_df)} sygnalow z pliku {signals_cache_file}.")
else:
    signals_df = mackowe_sygnaly(
        dfs=dfs_1w,
        settings=settings,
        require_vol_confirmation=True,
        require_cmo_confirmation=True,
        interval='1w',
        entry_offset=0,
        pattern_cols=['hammer', 'inverted_hammer', 'engulfing_bull', 'piercing_line'],
        debug=True
    )
    print("Zapisywanie sygnalow do pliku...")
    with open(signals_cache_file, "wb") as f:
        pickle.dump(signals_df, f)
    print("Sygnaly zapisane pomyslnie.")

signals_df.describe()


Znaleziono zapisane sygnaly. Wczytywanie z pliku...
Wczytano 912 sygnalow z pliku c:\Users\PC\Documents\Antigravity\lyse-lby\odbicie\cache\signals_cache_sp500.pkl.


,signal_time,entry_time,signal_close
count,912,912,912.000000
mean,2023-10-08 09:01:34.736842,2023-10-08 09:01:34.736842,129.103488
min,2021-07-25 00:00:00,2021-07-25 00:00:00,7.970000
25%,2022-05-15 00:00:00,2022-05-15 00:00:00,51.740002
50%,2023-10-08 00:00:00,2023-10-08 00:00:00,97.340000
75%,2025-03-16 00:00:00,2025-03-16 00:00:00,179.127495
max,2026-02-22 00:00:00,2026-02-22 00:00:00,565.369995
std,NaN,NaN,103.256636


## Wejście i Wyjście

In [4]:
# 3. Wybor Strategii Wejscia
# Zmien ta zmienna aby przełaczyc strategie: 'base', 'atr', 'bb'
STRATEGY = 'bb'

strat_settings = all_settings['strategies'][STRATEGY]['entry_settings']
tbm_settings = all_settings['strategies'][STRATEGY]['tbm_settings']

if STRATEGY == 'base':
    # --- Strategia bazowa: staly prog procentowy ---
    threshold_pct = strat_settings['threshold_pct']
    max_hold = strat_settings['max_setup_hold_bars']
    entries_df = generate_odbicie_entries(
        signals_df=signals_df,
        market_data_daily=dfs_1d,
        threshold_pct=threshold_pct,
        enter_on_close=strat_settings.get('enter_on_close', False),
        max_setup_hold_bars=max_hold
    )
    print(f"[base] Wygenerowano {len(entries_df)} wejsc przy progu {threshold_pct*100}%")

elif STRATEGY == 'atr':
    # --- Strategia ATR: prog oparty na wielokrotnosci ATR ---
    atr_period = strat_settings['atr_period']
    atr_factor = strat_settings['atr_factor']
    max_hold = strat_settings['max_setup_hold_bars']
    entries_df = generate_odbicie_atr_entries(
        signals_df=signals_df,
        market_data_daily=dfs_1d,
        atr_period=atr_period,
        atr_factor=atr_factor,
        enter_on_close=strat_settings.get('enter_on_close', False),
        max_setup_hold_bars=max_hold
    )
    print(f"[atr] Wygenerowano {len(entries_df)} wejsc (period={atr_period}, factor={atr_factor})")

elif STRATEGY == 'bb':
    # --- Strategia BB: wejscie przy dotknięciu dolnej wstegi Bollingera ---
    bb_period = strat_settings['bb_period']
    bb_std   = strat_settings['bb_std']
    max_hold = strat_settings['max_setup_hold_bars']
    entries_df = generate_odbicie_bb_entries(
        signals_df=signals_df,
        market_data_daily=dfs_1d,
        bb_period=bb_period,
        bb_std=bb_std,
        enter_on_close=strat_settings.get('enter_on_close', False),
        max_setup_hold_bars=max_hold
    )
    print(f"[bb] Wygenerowano {len(entries_df)} wejsc (period={bb_period}, std={bb_std})")

else:
    raise ValueError(f"Nieznana strategia: '{STRATEGY}'. Uzyj 'base', 'atr' lub 'bb'.")

entries_df.head()

[bb] Wygenerowano 239 wejsc (period=7, std=2.3)


,symbol,signal_time,pattern,entry_time,entry_price,signal_close,bb_period,bb_std,bb_lower,bb_middle,bb_upper,bb_bandwidth,rsi_at_entry,entry_atr,setup_bars
0,AOS,2022-02-06,inverted_hammer,2022-02-10,72.230003,73.580002,7,2.3,71.983393,73.971429,75.959465,0.053751,31.043324,2.154285,4
1,ACN,2025-04-13,inverted_hammer,2025-04-21,279.230011,284.339996,7,2.3,277.337949,284.975717,292.613484,0.053603,33.142228,10.259332,5
2,AES,2023-03-19,inverted_hammer,2023-03-24,22.209999,22.389999,7,2.3,21.545910,22.522857,23.499803,0.086752,31.657501,0.718662,5
3,A,2023-03-19,inverted_hammer,2023-03-23,131.130005,133.179993,7,2.3,129.964181,134.348570,138.732958,0.065269,35.100758,3.829334,4
4,ALB,2023-10-29,inverted_hammer,2023-10-30,127.410004,134.440002,7,2.3,126.938879,135.225717,143.512555,0.122563,26.233313,7.198791,1


In [5]:
# 4. Wyjście z użyciem Moving Triple Barrier Method
trades_df = moving_triple_barrier_labels(
    entries_df=entries_df,
    market_data_daily=dfs_1d,
    tp_mult=tbm_settings['tp_mult'],
    sl_mult=tbm_settings['sl_mult'],
    tp_trail_mult=tbm_settings['tp_trail_mult'],
    max_holding_bars=tbm_settings['max_holding_bars'],
    time_decay_sl=tbm_settings['time_decay_sl'],
    time_decay_mult=tbm_settings.get('time_decay_mult', 1.0),
    active_trailing_sl=tbm_settings['active_trail_sl'],
    sl_trail_mult=tbm_settings['sl_trail_mult'],
    exit_on_close=tbm_settings.get('exit_on_close', True)
)
print(f"Zakończono {len(trades_df)} transakcji.")
trades_df.head()


Zakończono 239 transakcji.


,symbol,signal_time,pattern,entry_time,entry_price,signal_close,bb_period,bb_std,bb_lower,bb_middle,...,entry_atr,setup_bars,exit_time,exit_price,return_pct,exit_reason,hold_bars,current_sl,current_tp_stop,is_tp_trailing
0,AOS,2022-02-06,inverted_hammer,2022-02-10,72.230003,73.580002,7,2.3,71.983393,73.971429,...,2.154285,4,2022-02-23,67.889999,-6.008589,TRAILING_SL,8,NaN,NaN,NaN
1,ACN,2025-04-13,inverted_hammer,2025-04-21,279.230011,284.339996,7,2.3,277.337949,284.975717,...,10.259332,5,2025-05-06,303.799988,8.799189,TRAILING_TP,11,NaN,NaN,NaN
2,AES,2023-03-19,inverted_hammer,2023-03-24,22.209999,22.389999,7,2.3,21.545910,22.522857,...,0.718662,5,2023-04-03,23.820000,7.248990,TRAILING_TP,6,NaN,NaN,NaN
3,A,2023-03-19,inverted_hammer,2023-03-23,131.130005,133.179993,7,2.3,129.964181,134.348570,...,3.829334,4,2023-04-05,138.089996,5.307703,TRAILING_TP,9,NaN,NaN,NaN
4,ALB,2023-10-29,inverted_hammer,2023-10-30,127.410004,134.440002,7,2.3,126.938879,135.225717,...,7.198791,1,2023-11-09,114.599998,-10.054160,TRAILING_SL,8,NaN,NaN,NaN


## Analiza

In [6]:
# 5. Analiza i Statystyki
if not trades_df.empty:
    wins = (trades_df['return_pct'] > 0).sum()
    losses = (trades_df['return_pct'] <= 0).sum()
    win_rate = wins / len(trades_df) * 100
    trades_df['return_per_bar'] = trades_df['return_pct'] / trades_df['hold_bars']

    print(f"Total Trades: {len(trades_df)}")
    print(f"Win Rate: {win_rate:.2f}%")
    print(f"Avg Return: {trades_df['return_pct'].mean():.2f}%")
    print(f"Avg bars held: {trades_df['hold_bars'].mean():.2f}")
    print(f"Avg Return per Bar: {trades_df['return_per_bar'].mean():.2f}%")


    # Powody wyjĹ›cia
    print("\nExit Reasons:")
    print(trades_df['exit_reason'].value_counts())
else:
    print("Brak transakcji do analizy.")

Total Trades: 239
Win Rate: 64.02%
Avg Return: 3.51%
Avg bars held: 9.10
Avg Return per Bar: 0.36%

Exit Reasons:
exit_reason
TRAILING_TP    130
TRAILING_SL     79
TIME_EXIT       27
OPEN             2
SL               1
Name: count, dtype: int64


In [7]:
temp = pd.DataFrame({
    'count': trades_df.groupby('exit_reason').return_pct.count(),
    'avg_return': trades_df.groupby('exit_reason').return_pct.mean(),
    'avg per Bar': trades_df.groupby('exit_reason').return_per_bar.mean(),
    'std per Bar': trades_df.groupby('exit_reason').return_per_bar.std(),
    'cumulativ_return': trades_df.groupby('exit_reason').return_pct.sum(),
    'std': trades_df.groupby('exit_reason').return_pct.std(),
    'avg_hold_bars': trades_df.groupby('exit_reason').hold_bars.mean(),
    'std_hold_bars': trades_df.groupby('exit_reason').hold_bars.std(),
    })

temp

,count,avg_return,avg per Bar,std per Bar,cumulativ_return,std,avg_hold_bars,std_hold_bars
exit_reason,,,,,,,,
OPEN,0,NaN,NaN,NaN,0.000000,NaN,5.500000,0.707107
SL,1,-17.052492,-17.052492,NaN,-17.052492,NaN,1.000000,NaN
TIME_EXIT,27,5.001070,0.333405,0.444508,135.028899,6.667617,15.000000,0.000000
TRAILING_SL,79,-5.261538,-0.855113,0.867418,-415.661482,3.216192,9.037975,3.538952
TRAILING_TP,130,8.688370,1.241689,0.981760,1129.488069,6.489339,8.038462,3.068732


## Ploty

In [8]:
# 6. Interaktywna Wizualizacja Transakcji
from odbicie.plot import show_trade_viewer
import ipywidgets as widgets
from IPython.display import display, clear_output

# Inicjalny rysunek
show_trade_viewer(
    trades_df,
    dfs_1d,
    tp_mult=tbm_settings['tp_mult'],
    sl_mult=tbm_settings['sl_mult'],
    ttp_mult=tbm_settings['tp_trail_mult'],
    max_holding_bars=tbm_settings['max_holding_bars'],
    exit_reason='All',
    active_trailing_sl=tbm_settings['active_trail_sl'],
    sl_trail_mult=tbm_settings['sl_trail_mult'],
    time_decay_sl=tbm_settings['time_decay_sl'],
    time_decay_mult=tbm_settings.get('time_decay_mult', 1.0),
    exit_on_close=tbm_settings.get('exit_on_close', True),
    strategy_type="tbm"
)

Output()

## Optymalizacja

### Optuna => BB

In [ ]:
def objective_bb_tbm_direct(trial):
    # Optymalizacja Parametrow BB
    bb_period = trial.suggest_int("bb_period", 5, 20)
    bb_std = trial.suggest_float("bb_std", 1.0, 4.0, step=0.1)
    max_setup_hold_bars = trial.suggest_int("max_setup_hold_bars", 5, 15)

    # Optymalizacja Wyjsc (TBM)
    tp_mult = trial.suggest_float("tp_mult", 0.2, 2.0, step=0.2)
    sl_mult = trial.suggest_float("sl_mult", 0.6, 3.0, step=0.2)
    tp_trail_mult = trial.suggest_float("tp_trail_mult", 0.01, 0.2, step=0.01)
    sl_trail_mult = trial.suggest_float("sl_trail_mult", 0.5, 5.0)
    time_decay_sl = trial.suggest_categorical("time_decay_sl", [True, False])
    time_decay_mult = trial.suggest_float("time_decay_mult", 0.5, 3.0)
    active_trailing_sl =  trial.suggest_categorical("trailing_sl", [True, False])
    max_holding_bars = trial.suggest_int("max_holding_bars", 5, 20)

    bb_tbm_settings = all_settings['strategies']['bb']['tbm_settings']

    # 1. Generowanie wejsc
    entries_df = generate_odbicie_bb_entries(
        signals_df=signals_df,
        market_data_daily=dfs_1d,
        bb_period=bb_period,
        bb_std=bb_std,
        enter_on_close=strat_settings.get('enter_on_close', False),
        rsi_period=14,
        max_setup_hold_bars=max_setup_hold_bars,
    )

    if entries_df.empty or len(entries_df) < 10:
        return 0.0

    # 2. Ewaluacja TBM
    trds = moving_triple_barrier_labels(
        entries_df=entries_df,
        market_data_daily=dfs_1d,
        tp_mult=tp_mult,
        sl_mult=sl_mult,
        tp_trail_mult=tp_trail_mult,
        max_holding_bars=max_holding_bars,
        time_decay_sl=time_decay_sl,
        time_decay_mult=time_decay_mult,
        active_trailing_sl=active_trailing_sl,
        sl_trail_mult=sl_trail_mult,
        exit_on_close=bb_tbm_settings.get('exit_on_close', True)
    )

    if len(trds) < 10:
        return 0.0

    avg_hold_bars = trds['hold_bars'].mean()
    if avg_hold_bars == 0:
        return 0.0

    avg_return = trds['return_pct'].mean()
    return avg_return / avg_hold_bars  # return_per_bar


'''
study_bb_tbm_direct = optuna.create_study(direction="maximize")
study_bb_tbm_direct.optimize(objective_bb_tbm_direct, n_trials=50)

print("Best parameters:", study_bb_tbm_direct.best_params)
print("Best value:", study_bb_tbm_direct.best_value)
'''


[I 2026-03-28 21:41:12,690] A new study created in memory with name: no-name-8e5dd7a9-aeef-48fe-9c23-a9fea0a2bd42
[I 2026-03-28 21:41:18,343] Trial 0 finished with value: -0.03574365670491159 and parameters: {'bb_period': 15, 'bb_std': 2.1, 'max_setup_hold_bars': 15, 'tp_mult': 0.6000000000000001, 'sl_mult': 1.4, 'tp_trail_mult': 0.2, 'sl_trail_mult': 2.8096203823053623, 'time_decay_sl': False, 'time_decay_mult': 2.4981260472718265, 'trailing_sl': False, 'max_holding_bars': 15}. Best is trial 0 with value: -0.03574365670491159.
[I 2026-03-28 21:41:24,423] Trial 1 finished with value: 0.12648941001202754 and parameters: {'bb_period': 7, 'bb_std': 2.4000000000000004, 'max_setup_hold_bars': 15, 'tp_mult': 0.6000000000000001, 'sl_mult': 2.6, 'tp_trail_mult': 0.04, 'sl_trail_mult': 3.8810130930395212, 'time_decay_sl': False, 'time_decay_mult': 1.2985299794649434, 'trailing_sl': True, 'max_holding_bars': 18}. Best is trial 1 with value: 0.12648941001202754.
[I 2026-03-28 21:41:30,341] Trial 

Best parameters: {'bb_period': 19, 'bb_std': 2.8, 'max_setup_hold_bars': 6, 'tp_mult': 0.4, 'sl_mult': 2.8000000000000003, 'tp_trail_mult': 0.17, 'sl_trail_mult': 2.6863086483522656, 'time_decay_sl': True, 'time_decay_mult': 1.4526506079599446, 'trailing_sl': False, 'max_holding_bars': 12}
Best value: 0.6204418790458058


### Optuna => ATR

In [10]:
def objective_atr_tbm_direct(trial):
    # Optymalizacja Parametrow ATR
    atr_period = trial.suggest_int("atr_period", 15, 20)
    atr_factor = trial.suggest_float("atr_factor", 2.0, 5.0)
    max_setup_hold_bars = trial.suggest_int("max_setup_hold_bars", 5, 15)

    # Optymalizacja Wyjsc (TBM)
    tp_mult = trial.suggest_float("tp_mult", 0.2, 2.0, step=0.2)
    sl_mult = trial.suggest_float("sl_mult", 0.6, 5.0, step=0.2)
    tp_trail_mult = trial.suggest_float("tp_trail_mult", 0.01, 0.2, step=0.01)
    sl_trail_mult = trial.suggest_float("sl_trail_mult", 0.5, 5.0)
    time_decay_sl = trial.suggest_categorical("time_decay_sl", [True, False])
    time_decay_mult = trial.suggest_float("time_decay_mult", 0.1, 3.0)
    active_trailing_sl =  trial.suggest_categorical("trailing_sl", [True, False])
    max_holding_bars = trial.suggest_int("max_holding_bars", 5, 25)

    atr_tbm_settings = all_settings['strategies']['atr']['tbm_settings']

    # 1. Generowanie wejsc
    entries_df = generate_odbicie_atr_entries(
        signals_df=signals_df,
        market_data_daily=dfs_1d,
        atr_period=atr_period,
        atr_factor=atr_factor,
        enter_on_close=strat_settings.get('enter_on_close', True),
        max_setup_hold_bars=max_setup_hold_bars,
    )

    if entries_df.empty or len(entries_df) < 10:
        return 0.0

    # 2. Ewaluacja TBM
    trds = moving_triple_barrier_labels(
        entries_df=entries_df,
        market_data_daily=dfs_1d,
        tp_mult=tp_mult,
        sl_mult=sl_mult,
        tp_trail_mult=tp_trail_mult,
        max_holding_bars=max_holding_bars,
        time_decay_sl=time_decay_sl,
        time_decay_mult=time_decay_mult,
        active_trailing_sl=active_trailing_sl,
        sl_trail_mult=sl_trail_mult,
        exit_on_close=atr_tbm_settings.get('exit_on_close', True)
    )

    if len(trds) < 10:
        return 0.0

    avg_hold_bars = trds['hold_bars'].mean()
    if avg_hold_bars == 0:
        return 0.0

    avg_return = trds['return_pct'].mean()
    return avg_return / avg_hold_bars  # return_per_bar

'''
study_atr_tbm_direct = optuna.create_study(direction="maximize")
study_atr_tbm_direct.optimize(objective_atr_tbm_direct, n_trials=50)

print("Best parameters:", study_atr_tbm_direct.best_params)
print("Best value:", study_atr_tbm_direct.best_value)

'''

'\nstudy_atr_tbm_direct = optuna.create_study(direction="maximize")\nstudy_atr_tbm_direct.optimize(objective_atr_tbm_direct, n_trials=50)\n\nprint("Best parameters:", study_atr_tbm_direct.best_params)\nprint("Best value:", study_atr_tbm_direct.best_value)\n\n'

### Optuna => base

In [ ]:
def objective_base_tbm_direct(trial):
    # Optymalizacja Parametrow base
    threshold_pct = trial.suggest_float("threshold_pct", 0.05, 0.5)
    max_setup_hold_bars = trial.suggest_int("max_setup_hold_bars", 5, 15)

    # Optymalizacja Wyjsc (TBM)
    tp_mult = trial.suggest_float("tp_mult", 0.2, 2.0, step=0.2)
    sl_mult = trial.suggest_float("sl_mult", 0.6, 3.0, step=0.2)
    tp_trail_mult = trial.suggest_float("tp_trail_mult", 0.01, 0.2, step=0.01)
    sl_trail_mult = trial.suggest_float("sl_trail_mult", 0.5, 5.0)
    time_decay_sl = trial.suggest_categorical("time_decay_sl", [True, False])
    time_decay_mult = trial.suggest_float("time_decay_mult", 0.5, 3.0)
    active_trailing_sl =  trial.suggest_categorical("trailing_sl", [True, False])
    max_holding_bars = trial.suggest_int("max_holding_bars", 5, 20)

    base_tbm_settings = all_settings['strategies']['base']['tbm_settings']

    # 1. Generowanie wejsc
    entries_df = generate_odbicie_entries(
        signals_df=signals_df,
        market_data_daily=dfs_1d,
        threshold_pct=threshold_pct,
        enter_on_close=strat_settings.get('enter_on_close', True),
        max_setup_hold_bars=max_setup_hold_bars,
    )

    if entries_df.empty or len(entries_df) < 10:
        return 0.0

    # 2. Ewaluacja TBM
    trds = moving_triple_barrier_labels(
        entries_df=entries_df,
        market_data_daily=dfs_1d,
        tp_mult=tp_mult,
        sl_mult=sl_mult,
        tp_trail_mult=tp_trail_mult,
        max_holding_bars=max_holding_bars,
        time_decay_sl=time_decay_sl,
        time_decay_mult=time_decay_mult,
        active_trailing_sl=active_trailing_sl,
        sl_trail_mult=sl_trail_mult,
        exit_on_close=base_tbm_settings.get('exit_on_close', True)
    )

    if len(trds) < 10:
        return 0.0

    avg_hold_bars = trds['hold_bars'].mean()
    if avg_hold_bars == 0:
        return 0.0

    avg_return = trds['return_pct'].mean()
    return avg_return / avg_hold_bars  # return_per_bar

'''
study_base_tbm_direct = optuna.create_study(direction="maximize")
study_base_tbm_direct.optimize(objective_base_tbm_direct, n_trials=50)

print("Best parameters:", study_base_tbm_direct.best_params)
print("Best value:", study_base_tbm_direct.best_value)
'''

[I 2026-03-28 21:45:30,437] A new study created in memory with name: no-name-3b8b756a-0402-44c0-9e28-90ad23526d23
[I 2026-03-28 21:45:33,201] Trial 0 finished with value: 0.24956124756814735 and parameters: {'threshold_pct': 0.09212480188183822, 'max_setup_hold_bars': 13, 'tp_mult': 2.0, 'sl_mult': 2.2, 'tp_trail_mult': 0.2, 'sl_trail_mult': 0.5106320314222976, 'time_decay_sl': False, 'time_decay_mult': 1.8655418406012823, 'trailing_sl': True, 'max_holding_bars': 15}. Best is trial 0 with value: 0.24956124756814735.
[I 2026-03-28 21:45:36,136] Trial 1 finished with value: 0.19430822507462783 and parameters: {'threshold_pct': 0.05494471793874902, 'max_setup_hold_bars': 6, 'tp_mult': 1.2, 'sl_mult': 2.6, 'tp_trail_mult': 0.11, 'sl_trail_mult': 0.569524605580273, 'time_decay_sl': False, 'time_decay_mult': 1.369608562275774, 'trailing_sl': True, 'max_holding_bars': 10}. Best is trial 0 with value: 0.24956124756814735.
[I 2026-03-28 21:45:38,363] Trial 2 finished with value: 0.1603686546291

Best parameters: {'threshold_pct': 0.22208512032414665, 'max_setup_hold_bars': 12, 'tp_mult': 1.4000000000000001, 'sl_mult': 2.6, 'tp_trail_mult': 0.12, 'sl_trail_mult': 2.735818312293845, 'time_decay_sl': True, 'time_decay_mult': 2.308431729479876, 'trailing_sl': False, 'max_holding_bars': 20}
Best value: 0.8423869463540896


## Lookup

### Model
Leave-one-out cross-validation of the meta-learning model. Shows how accurately the model predicts each of the 9 TBM parameters from historical data.

In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyArrowPatch

# ─── PREDICTOR SWITCH ───────────────────────────────────────────
# Change 'xgb' to 'mtl' to use the PyTorch neural network instead
PREDICTOR_TYPE = 'mtl'   # 'xgb' | 'mtl'

# ────────────────────────────────────────────────────────────────
LOOKUP_PATH = "cache/lookup/tbm_lookup.json"
STRATEGY    = "base"   # 'base' | 'atr' | 'bb'
DANE_DIR    = "cache"

from odbicie.tbm.predyktor.pred_factory import get_predictor
predictor = get_predictor(PREDICTOR_TYPE, LOOKUP_PATH, STRATEGY, cache_dir=os.path.join(DANE_DIR, 'predyktor'))
print(predictor)


if not predictor.is_ready:
    print(f'Not enough data for strategy={STRATEGY!r} (need >= 15 records). Run tbm_settings_generator first.')
else:
    print(f'Running leave-one-out validation on {len(predictor._records)} records...')
    print('(This may take ~30s depending on dataset size)')
    loo = predictor.leave_one_out_errors()
    print('Done!')

    reg_targets = loo['reg_targets']   # 6 continuous/integer
    cls_targets = loo['cls_targets']   # 2 booleans
    reg_errors  = loo['reg_errors']    # shape (n, 6)
    cls_errors  = loo['cls_errors']    # shape (n, 2)
    Y_reg       = loo['Y_reg']
    Y_cls       = loo['Y_cls']
    n           = loo['n']

    # ── Styling ───────────────────────────────────────────────────────────
    plt.style.use('dark_background')
    ACCENT   = '#7EB8F7'   # blue
    ACCENT2  = '#F7C27E'   # amber
    GOOD     = '#5CDB95'   # green
    BAD      = '#FC4F4F'   # red
    BG       = '#141824'
    PANEL    = '#1E2436'

    # ══════════════════════════════════════════════════════════════════════
    # PLOT 1 — MAE bar chart for all 9 targets
    # ══════════════════════════════════════════════════════════════════════
    fig, ax = plt.subplots(figsize=(13, 5), facecolor=BG)
    ax.set_facecolor(PANEL)

    mae_reg = reg_errors.mean(axis=0)
    mae_cls = cls_errors.mean(axis=0)   # fraction wrong (0-1)

    all_labels = reg_targets + cls_targets
    all_mae    = list(mae_reg) + list(mae_cls)
    colors     = [ACCENT] * len(reg_targets) + [ACCENT2] * len(cls_targets)

    bars = ax.bar(all_labels, all_mae, color=colors, edgecolor='none', width=0.6, zorder=3)
    ax.grid(axis='y', color='white', alpha=0.07, zorder=0)
    ax.set_ylabel('Mean Absolute Error (LOO)', color='white', fontsize=11)
    ax.set_title(f'Predictor Quality — Leave-One-Out MAE per TBM Parameter  |  strategy={STRATEGY!r}  |  n={n}',
                 color='white', fontsize=13, fontweight='bold', pad=12)
    ax.tick_params(colors='white'); ax.spines[:].set_visible(False)
    ax.set_ylim(bottom=0)

    # Value labels on bars
    for bar, val, is_cls in zip(bars, all_mae, [False]*6 + [True]*2):
        label = f'{val:.1%}' if is_cls else f'{val:.3f}'
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(all_mae)*0.01,
                label, ha='center', va='bottom', color='white', fontsize=9)

    # Legend patch
    import matplotlib.patches as mpatches
    ax.legend(handles=[
        mpatches.Patch(color=ACCENT,  label='Regression targets (continuous / integer)'),
        mpatches.Patch(color=ACCENT2, label='Classification targets (boolean — shown as error rate)'),
    ], facecolor=BG, edgecolor='none', labelcolor='white', fontsize=9, loc='upper right')

    plt.tight_layout()
    plt.show()

    # ══════════════════════════════════════════════════════════════════════
    # PLOT 2 — Actual vs Predicted scatter for each regression target
    # ══════════════════════════════════════════════════════════════════════
    # Reconstruct LOO predictions from errors (pred = actual - signed_error)
    # We only have absolute errors so we'll display error distribution instead
    ncols = 3
    nrows = 2
    fig2, axes = plt.subplots(nrows, ncols, figsize=(14, 8), facecolor=BG)
    fig2.suptitle(f'LOO Error Distributions — Regression Targets  |  strategy={STRATEGY!r}',
                  color='white', fontsize=13, fontweight='bold', y=1.01)

    for idx, (tgt, errors_col, actuals_col) in enumerate(
            zip(reg_targets, reg_errors.T, Y_reg.T)):
        row, col = divmod(idx, ncols)
        ax2 = axes[row][col]
        ax2.set_facecolor(PANEL)

        # Violin + jitter scatter
        parts = ax2.violinplot(errors_col, positions=[0], widths=0.6,
                               showmedians=True, showextrema=False)
        for pc in parts['bodies']:
            pc.set_facecolor(ACCENT); pc.set_alpha(0.4)
        parts['cmedians'].set_color(GOOD); parts['cmedians'].set_linewidth(2)

        jitter = np.random.uniform(-0.15, 0.15, size=len(errors_col))
        ax2.scatter(jitter, errors_col, color=ACCENT, alpha=0.5, s=18, zorder=3)

        mae_val = errors_col.mean()
        ax2.axhline(mae_val, color=ACCENT2, linestyle='--', linewidth=1.2,
                    label=f'MAE={mae_val:.3f}')

        ax2.set_title(tgt, color='white', fontsize=10, fontweight='bold')
        ax2.set_ylabel('Absolute Error', color='white', fontsize=8)
        ax2.set_xticks([])
        ax2.tick_params(colors='white', labelsize=8)
        ax2.spines[:].set_visible(False)
        ax2.grid(axis='y', color='white', alpha=0.07)
        ax2.legend(facecolor=BG, edgecolor='none', labelcolor='white', fontsize=8)

    plt.tight_layout()
    plt.show()

    # ══════════════════════════════════════════════════════════════════════
    # PLOT 3 — Boolean classification accuracy bars
    # ══════════════════════════════════════════════════════════════════════
    fig3, axes3 = plt.subplots(1, 2, figsize=(9, 4), facecolor=BG)
    fig3.suptitle(f'LOO Boolean Prediction Accuracy  |  strategy={STRATEGY!r}',
                  color='white', fontsize=13, fontweight='bold')

    for idx, (tgt, err_col, actual_col) in enumerate(
            zip(cls_targets, cls_errors.T, Y_cls.T)):
        ax3 = axes3[idx]
        ax3.set_facecolor(PANEL)

        error_rate = err_col.mean()
        accuracy   = 1 - error_rate

        # Stacked bar: accuracy (green) + error (red)
        ax3.bar(['Prediction'], [accuracy], color=GOOD, label=f'Correct ({accuracy:.1%})', width=0.4)
        ax3.bar(['Prediction'], [error_rate], bottom=[accuracy], color=BAD,
                label=f'Wrong ({error_rate:.1%})', width=0.4)

        # Class distribution
        true_rate  = actual_col.mean()
        false_rate = 1 - true_rate
        ax3.bar(['Actual dist.'], [true_rate],  color=ACCENT,  label=f'True  ({true_rate:.1%})',  width=0.4)
        ax3.bar(['Actual dist.'], [false_rate], bottom=[true_rate], color=ACCENT2,
                label=f'False ({false_rate:.1%})', width=0.4)

        ax3.set_title(tgt, color='white', fontsize=11, fontweight='bold')
        ax3.set_ylim(0, 1.05)
        ax3.set_ylabel('Fraction', color='white')
        ax3.tick_params(colors='white'); ax3.spines[:].set_visible(False)
        ax3.grid(axis='y', color='white', alpha=0.07)
        ax3.legend(facecolor=BG, edgecolor='none', labelcolor='white', fontsize=8, loc='lower right')

    plt.tight_layout()
    plt.show()

    # ══════════════════════════════════════════════════════════════════════
    # PLOT 4 — Error heatmap (records × targets)
    # ══════════════════════════════════════════════════════════════════════
    # Normalise each column by its mean so all targets are on the same 0-3 scale
    reg_norm = reg_errors / (reg_errors.mean(axis=0) + 1e-9)
    cls_norm = cls_errors.astype(float)          # already 0/1
    heat     = np.hstack([reg_norm, cls_norm])

    fig4, ax4 = plt.subplots(figsize=(13, max(4, n * 0.18 + 1)), facecolor=BG)
    ax4.set_facecolor(BG)

    im = ax4.imshow(heat, aspect='auto', cmap='RdYlGn_r', vmin=0, vmax=2.5,
                    interpolation='nearest')

    ax4.set_xticks(range(len(all_labels)))
    ax4.set_xticklabels(all_labels, rotation=30, ha='right', color='white', fontsize=9)
    ax4.set_yticks([])
    ax4.set_ylabel(f'Records (n={n})', color='white', fontsize=10)
    ax4.set_title(
        f'LOO Error Heatmap (normalised per column)  |  strategy={STRATEGY!r}\n'
        'Green = easy to predict  |  Red = hard to predict',
        color='white', fontsize=12, fontweight='bold', pad=10)
    ax4.spines[:].set_visible(False)

    cbar = fig4.colorbar(im, ax=ax4, shrink=0.6, pad=0.02)
    cbar.ax.tick_params(colors='white')
    cbar.set_label('Normalised error (1 = MAE)', color='white', fontsize=9)

    plt.tight_layout()
    plt.show()

    # ── Summary stats —————————————————————————————————————————————
    print('\n── LOO Summary ──────────────────────────────────────────────────')
    print(f'{"Target":<20} {"MAE":>10} {"Median AE":>12} {"Max AE":>10}')
    print('-' * 55)
    for tgt, errs in zip(reg_targets, reg_errors.T):
        print(f'{tgt:<20} {errs.mean():>10.4f} {np.median(errs):>12.4f} {errs.max():>10.4f}')
    for tgt, errs in zip(cls_targets, cls_errors.T):
        accuracy = 1 - errs.mean()
        print(f'{tgt:<20} {"accuracy":>10}  {accuracy:>11.1%}')


ModuleNotFoundError: No module named 'tbm.tbm_predictor_factory'

### Comparison

In [29]:
import pandas as pd
import os
import json
from odbicie.tbm.predyktor.pred_factory import get_predictor

# 1. Manually resolve paths to be safe
# We assume you are running inside 'odbicie' folder or project root
CWD = os.getcwd()
BASE_DIR = CWD if os.path.basename(CWD) == 'odbicie' else os.path.join(CWD, 'odbicie')
CACHE_DIR = os.path.join(BASE_DIR, 'cache')
LOOKUP_PATH = os.path.join(CACHE_DIR, 'lookup', 'tbm_lookup.json')
SETTINGS_PATH = os.path.join(BASE_DIR, 'settings.json')

# 2. Load settings and choose STRATEGY
with open(SETTINGS_PATH, 'r') as f:
    all_settings = json.load(f)

# Choose strategy ('bb', 'atr', or 'base')
CURRENT_STRAT = 'base' 
strat_settings = all_settings['strategies'][CURRENT_STRAT]['entry_settings']

# 3. Setup Parameters for the test
if CURRENT_STRAT == 'bb':
    entry_params = {
        'bb_period': strat_settings['bb_period'],
        'bb_std': strat_settings['bb_std'],
        'max_setup_hold_bars': strat_settings['max_setup_hold_bars']
    }
elif CURRENT_STRAT == 'atr':
    entry_params = {
        'atr_period': strat_settings['atr_period'],
        'atr_factor': strat_settings['atr_factor'],
        'max_setup_hold_bars': strat_settings['max_setup_hold_bars']
    }
else:
    entry_params = {
        'threshold_pct': strat_settings['threshold_pct'],
        'max_setup_hold_bars': strat_settings['max_setup_hold_bars']
    }

context_features = {'avg_rsi_at_entry': 30.0, 'avg_atr_pct': 0.02, 'avg_setup_bars': 5.0, 'entry_count_log': 3.0}

# 4. Get Predictions and Optuna Match
results = {}
for ptype in ['xgb', 'mtl']:
    pred = get_predictor(ptype, LOOKUP_PATH, CURRENT_STRAT, cache_dir=os.path.join(CACHE_DIR, 'predyktor'))
    name = ptype.upper()
    if pred.is_ready:
        results[name] = pred.predict(entry_params, context_features)
    else:
        results[name] = {k: f"Need {15 - len(pred._records)} more records" for k in ["tp_mult", "sl_mult"]}

# Try finding Optuna Match
tmp_pred = get_predictor('xgb', LOOKUP_PATH, CURRENT_STRAT, cache_dir=os.path.join(CACHE_DIR, 'predyktor'))
neighbors = tmp_pred.knn_neighbors(entry_params, k=1)
if neighbors and neighbors[0]['distance'] < 1e-5:
    results['OPTUNA (Actual)'] = neighbors[0]['tbm_params']
else:
    results['OPTUNA (Actual)'] = {k: "No direct match" for k in ["tp_mult", "sl_mult"]}

# 5. Output
print(f"Comparing Predictors for strategy: {CURRENT_STRAT}")
display(pd.DataFrame.from_dict(results))


[TbmPredictor] Trained on 174 records for strategy='base' (6 features -> 6 reg + 2 cls targets)
[TbmPredictorMTL] Trained on 174 records for strategy='base' (6 features -> 6 reg + 2 cls heads)
Comparing Predictors for strategy: base


,XGB,MTL,OPTUNA (Actual)
tp_mult,2.30117,1.178531,4.4
sl_mult,5.887368,6.261729,7.4
tp_trail_mult,0.278684,0.148775,0.56
sl_trail_mult,3.249779,5.920358,9.2
time_decay_mult,4.165958,3.685022,4.7
max_holding_bars,6,21,5
active_trail_sl,False,True,True
time_decay_sl,False,False,False
exit_on_close,True,True,True
